# Query Expansion

You will generate alternative query phrasing to improve retrieval recall.


## Why Expansion Helps

You can express one intent with many word choices.

You will expand queries to reduce vocabulary mismatch.


In [ ]:
# DEPENDENCY: pip install -q openai pydantic
# (packages should be pre-installed in venv before running this script)

In [ ]:
import os
import json
from getpass import getpass
from openai import OpenAI

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-5-mini"

## Basic Expansion

You will request concise alternatives from the model.


In [ ]:
def expand_query(query: str, n_expansions: int = 3) -> list[str]:
    """Generate alternative phrasings of a search query."""
    
    prompt = f"""Generate {n_expansions} alternative search queries for the following query.
Each alternative should:
- Capture the same information need
- Use different keywords or phrasing
- Be concise (under 10 words)

Original query: {query}

Return ONLY a JSON array of strings, no explanation.
Example: ["alternative 1", "alternative 2", "alternative 3"]"""
    
    response = client.responses.create(
        model=MODEL,
        input=prompt
    )
    
    # Parse the JSON array
    try:
        expansions = json.loads(response.output_text)
        return expansions
    except json.JSONDecodeError:
        # Fallback: return original query
        return [query]

# Test it
original = "how to debug Python code"
expansions = expand_query(original)

print(f"Original: {original}")
print(f"Expansions:")
for i, exp in enumerate(expansions, 1):
    print(f"  {i}. {exp}")

## Structured Expansion

You will enforce schema based output.


In [ ]:
from pydantic import BaseModel, Field

class QueryExpansion(BaseModel):
    """Structured output for query expansion."""
    original: str = Field(description="The original query")
    synonyms: list[str] = Field(description="Queries using synonym terms")
    related_concepts: list[str] = Field(description="Queries for related concepts")
    specific: list[str] = Field(description="More specific versions of the query")
    general: list[str] = Field(description="More general versions of the query")


def expand_query_structured(query: str) -> QueryExpansion:
    """Generate structured query expansions."""
    
    prompt = f"""Expand the following search query into multiple alternative queries.

Original query: {query}

Generate:
- 2 queries using synonyms of the original terms
- 2 queries for related concepts
- 1 more specific query
- 1 more general query

Keep each query concise (under 10 words)."""

    # Get JSON schema from Pydantic model
    schema = QueryExpansion.model_json_schema()
    schema["additionalProperties"] = False
    
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": "QueryExpansion",
                "strict": True,
                "schema": schema
            }
        }
    )
    
    return QueryExpansion.model_validate_json(response.output_text)


# Test structured expansion
expansion = expand_query_structured("machine learning best practices")

print(f"Original: {expansion.original}")
print(f"\nSynonyms:")
for q in expansion.synonyms:
    print(f"  - {q}")
print(f"\nRelated Concepts:")
for q in expansion.related_concepts:
    print(f"  - {q}")
print(f"\nMore Specific:")
for q in expansion.specific:
    print(f"  - {q}")
print(f"\nMore General:")
for q in expansion.general:
    print(f"  - {q}")

## Weighted Expansion

You will keep stronger weight on the original query.


In [ ]:
from dataclasses import dataclass

@dataclass
class WeightedQuery:
    text: str
    weight: float

def expand_with_weights(query: str, original_weight: float = 2.0) -> list[WeightedQuery]:
    """Expand query and return weighted query list."""
    
    # Start with the original query at higher weight
    weighted_queries = [WeightedQuery(query, original_weight)]
    
    # Get expansions
    expansions = expand_query(query, n_expansions=2)
    
    # Add expansions at weight 1.0
    for exp in expansions:
        weighted_queries.append(WeightedQuery(exp, 1.0))
    
    return weighted_queries

# Test
weighted = expand_with_weights("Python async programming")
print("Weighted queries:")
for wq in weighted:
    print(f"  [{wq.weight:.1f}x] {wq.text}")

## Context Aware Expansion

You will condition expansions on domain context.


In [ ]:
def expand_query_with_context(query: str, context: str, n_expansions: int = 3) -> list[str]:
    """Generate context-aware query expansions."""
    
    prompt = f"""Generate {n_expansions} alternative search queries.

Context: {context}
Original query: {query}

The alternatives should:
- Be relevant to the given context
- Capture the same information need
- Use domain-appropriate terminology

Return ONLY a JSON array of strings."""
    
    response = client.responses.create(
        model=MODEL,
        input=prompt
    )
    
    try:
        return json.loads(response.output_text)
    except json.JSONDecodeError:
        return [query]

# Compare expansions with different contexts
query = "memory management"

contexts = [
    "Software engineering and programming",
    "Cognitive psychology and learning",
    "Computer hardware and systems"
]

print(f"Original query: '{query}'\n")

for ctx in contexts:
    print(f"Context: {ctx}")
    expansions = expand_query_with_context(query, ctx, n_expansions=2)
    for exp in expansions:
        print(f"  - {exp}")
    print()

## Caching

You will cache expansions to reduce repeated latency and cost.


In [ ]:
import hashlib
import time

class CachedQueryExpander:
    """Query expander with in-memory caching."""
    
    def __init__(self):
        self.cache = {}
        self.hits = 0
        self.misses = 0
    
    def _cache_key(self, query: str) -> str:
        """Generate cache key from normalized query."""
        normalized = query.lower().strip()
        return hashlib.md5(normalized.encode()).hexdigest()
    
    def expand(self, query: str, n_expansions: int = 3) -> list[str]:
        """Expand query with caching."""
        key = self._cache_key(query)
        
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        
        self.misses += 1
        expansions = expand_query(query, n_expansions)
        self.cache[key] = expansions
        return expansions
    
    def stats(self) -> dict:
        """Return cache statistics."""
        total = self.hits + self.misses
        hit_rate = self.hits / total if total > 0 else 0
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": f"{hit_rate:.1%}",
            "cache_size": len(self.cache)
        }

# Test caching
expander = CachedQueryExpander()

queries = [
    "Python debugging",
    "python debugging",  # Same query, different case - should hit cache
    "JavaScript frameworks",
    "Python debugging",  # Repeat - should hit cache
]

for q in queries:
    start = time.time()
    result = expander.expand(q)
    elapsed = time.time() - start
    print(f"'{q}' -> {len(result)} expansions ({elapsed*1000:.0f}ms)")

print(f"\nCache stats: {expander.stats()}")

## Expansion Quality

You will evaluate semantic alignment and lexical diversity.


In [ ]:
def evaluate_expansion(original: str, expansion: str) -> dict:
    """Have the LLM evaluate an expansion's quality."""
    
    prompt = f"""Evaluate this query expansion on a scale of 1-5 for each criterion.

Original: {original}
Expansion: {expansion}

Criteria:
- semantic_similarity: Does the expansion capture the same information need? (1=unrelated, 5=identical intent)
- lexical_diversity: Does it use different words? (1=same words, 5=completely different vocabulary)
- specificity: Is it appropriately specific? (1=too vague/specific, 5=well-calibrated)

Return ONLY a JSON object with these three keys and integer scores."""

    response = client.responses.create(
        model=MODEL,
        input=prompt
    )
    
    try:
        return json.loads(response.output_text)
    except:
        return {"error": "Failed to parse evaluation"}

# Evaluate some expansions
original = "Python error handling"
test_expansions = [
    "Python exception management",  # Good - different words, same concept
    "Python try except blocks",      # Good - more specific
    "JavaScript error handling",     # Bad - wrong language
    "coding",                         # Bad - too vague
]

print(f"Original: '{original}'\n")
for exp in test_expansions:
    scores = evaluate_expansion(original, exp)
    print(f"'{exp}'")
    for key, value in scores.items():
        print(f"  {key}: {value}")
    print()

## Exercises With Starter and Solution

### Exercise 1

You will generate negative queries for meanings to exclude.


In [ ]:
# Starter code for Exercise 1
def generate_negative_queries(query: str, context: str = None) -> list[str]:
    # TODO generate negative queries that represent unwanted meanings
    pass


In [ ]:
# Solution for Exercise 1
def generate_negative_queries(query: str, context: str = None) -> list[str]:
    context_text = context if context else 'No extra context'
    prompt = f"""Create 3 short negative queries for this user query.
Negative queries represent meanings to exclude.

User query: {query}
Context: {context_text}

Return only a JSON array of strings."""

    response = client.responses.create(model=MODEL, input=prompt)
    try:
        items = json.loads(response.output_text)
        return [str(x) for x in items]
    except Exception:
        return []

print(generate_negative_queries('Python web frameworks', context='Software engineering'))


### Exercise 2

You will build iterative expansion from broad to specific.


In [ ]:
# Starter code for Exercise 2
def iterative_expansion(query: str, depth: int = 3) -> list[str]:
    # TODO create iterative broad to specific expansion chain
    pass


In [ ]:
# Solution for Exercise 2
def iterative_expansion(query: str, depth: int = 3) -> list[str]:
    chain = [query]
    current = query

    for _ in range(depth):
        prompt = f"""Refine this search query to a more specific version while preserving intent.
Current query: {current}
Return only the next query text."""
        response = client.responses.create(model=MODEL, input=prompt)
        next_query = response.output_text.strip().strip('"')
        if not next_query:
            break
        chain.append(next_query)
        current = next_query

    return chain

print(iterative_expansion('Python performance', depth=3))


### Exercise 3

You will generate multilingual expansions.


In [ ]:
# Starter code for Exercise 3
def multilingual_expansion(query: str, languages: list[str]) -> dict[str, list[str]]:
    # TODO generate two expansions per language
    pass


In [ ]:
# Solution for Exercise 3
def multilingual_expansion(query: str, languages: list[str]) -> dict[str, list[str]]:
    outputs = {}

    for language in languages:
        prompt = f"""Generate 2 alternative search queries in {language}.
Keep intent equal to this query: {query}
Return only a JSON array of strings."""

        response = client.responses.create(model=MODEL, input=prompt)

        try:
            outputs[language] = [str(x) for x in json.loads(response.output_text)]
        except Exception:
            outputs[language] = [query]

    return outputs

print(multilingual_expansion('deploy web app', ['Spanish', 'French']))


## What You Learned

You can now generate, weight, cache, and evaluate query expansions.
